# AI Tutor on Kaggle — Ollama + Cloudflare Quick Tunnel

Thin launcher: Ollama serves **Qwen2.5-7B-Instruct Q4_K_M** (chat) and **bge-m3** (embeddings); nginx on `127.0.0.1:7860` fronts the FastAPI backend and the frontend; a Cloudflare Quick Tunnel exposes nginx. No vLLM process is started and port 8001 is not used.

Enable a GPU accelerator and Internet, check the config cell, then use **Run All**.

* A warm **Run All** (same Kaggle runtime) skips apt/pip/npm/Ollama/cloudflared installs, model pulls, tests, and — when the code, config and services are unchanged and healthy — the service restart.
* State restoration is attempted only once per runtime, before services start; `data/` and `vectorstore/` are saved to a private Kaggle Dataset every 10 minutes.
* The `*.trycloudflare.com` URL changes every time a new tunnel is created. Google Calendar OAuth still uses the `GOOGLE_CALENDAR_REDIRECT_URI` / `AI_TUTOR_FRONTEND_URL` Kaggle Secrets unchanged.
* Cells under **Manual actions** do nothing during a normal Run All.

In [ ]:
# User configuration: edit values only in this cell.

REPOSITORY_URL = "https://github.com/qtrung123/AI_Tutor2-Kaggle.git"
REPOSITORY_BRANCH = "main"
PROJECT_ROOT = "/kaggle/working/AI_Tutor2"

UPDATE_CODE = True     # git fetch + fast-forward pull on every Run All.
QUICK_START = True     # Skips tests and unchanged pip/npm installs.
FORCE_RESTART = False  # True: restart all services even if they are healthy and unchanged.

GGUF_MODEL_PATH = ""
LECTURE_INPUT_DIR = ""

LLM_BACKEND = "ollama"
OLLAMA_CHAT_MODEL = "hf.co/bartowski/Qwen2.5-7B-Instruct-GGUF:Q4_K_M"
OLLAMA_DEFAULT_GENERATION_MODEL = "qwen-2.5-7b"
OLLAMA_GENERATION_MODELS = "qwen-2.5-7b"
PRELOAD_ALL_MODELS = False
OLLAMA_EMBEDDING_MODEL = "bge-m3"
RECREATE_OLLAMA_MODEL = False

BACKEND_PORT = 8000
FRONTEND_PORT = 3000
PUBLIC_PORT = 7860  # nginx; the Cloudflare tunnel points here, never at the backend port.

STATE_DATASET_SLUG = "trung121212/ai-tutor-state"
CACHE_DATASET_SLUG = ""
STATE_SNAPSHOT_INTERVAL_SECONDS = 10 * 60

import contextlib, os, time

APP_ENV = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "LLM_BACKEND": LLM_BACKEND,
    "GGUF_MODEL_PATH": GGUF_MODEL_PATH,
    "OLLAMA_CHAT_MODEL": OLLAMA_CHAT_MODEL,
    "OLLAMA_DEFAULT_GENERATION_MODEL": OLLAMA_DEFAULT_GENERATION_MODEL,
    "OLLAMA_GENERATION_MODELS": OLLAMA_GENERATION_MODELS,
    "PRELOAD_ALL_MODELS": str(PRELOAD_ALL_MODELS).lower(),
    "OLLAMA_EMBEDDING_MODEL": OLLAMA_EMBEDDING_MODEL,
    "RECREATE_OLLAMA_MODEL": int(RECREATE_OLLAMA_MODEL),
    "BACKEND_PORT": BACKEND_PORT,
    "FRONTEND_PORT": FRONTEND_PORT,
    "PUBLIC_PORT": PUBLIC_PORT,
    "REBUILD_CHROMA_ON_EMBEDDING_CHANGE": 1,
    "QUIZ_GENERATION_RETRY_LIMIT": 3,
}
for key, value in APP_ENV.items():
    os.environ[key] = str(value)

# Drop vLLM settings a previous notebook may have left in this kernel; this launcher never uses them.
for key in ("VLLM_MODEL", "VLLM_GPU_MEMORY_UTILIZATION", "VLLM_PORT", "VLLM_BASE_URL"):
    os.environ.pop(key, None)

RUN_STARTED = time.time()
STARTUP_TIMINGS = {}

@contextlib.contextmanager
def timed(stage):
    started = time.time()
    try:
        yield
    finally:
        STARTUP_TIMINGS[stage] = time.time() - started
        print(f"[timing] {stage}: {STARTUP_TIMINGS[stage]:.1f}s")

print("LLM_BACKEND =", os.environ["LLM_BACKEND"])
print("Chat model  =", OLLAMA_CHAT_MODEL)
print("Embeddings  =", OLLAMA_EMBEDDING_MODEL)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

google_secret_names = [
    "GOOGLE_CLIENT_ID",
    "GOOGLE_CLIENT_SECRET",
    "GOOGLE_CALENDAR_REDIRECT_URI",
    "GOOGLE_CALENDAR_TOKEN_ENCRYPTION_KEY",
    "AI_TUTOR_FRONTEND_URL",
]

secrets = UserSecretsClient()

for name in google_secret_names:
    value = secrets.get_secret(name)

    if not value:
        raise RuntimeError(f"Missing Kaggle Secret: {name}")

    os.environ[name] = value

print("Google Calendar configuration loaded:")
for name in google_secret_names:
    print(f"✓ {name}")

In [ ]:
from pathlib import Path
import subprocess

with timed("git"):
    root = Path(PROJECT_ROOT)
    if not (root / ".git").exists():
        root.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch", REPOSITORY_URL, PROJECT_ROOT], check=True)
    elif UPDATE_CODE:
        subprocess.run(["git", "-C", PROJECT_ROOT, "fetch", "origin", REPOSITORY_BRANCH], check=True)
        subprocess.run(["git", "-C", PROJECT_ROOT, "checkout", REPOSITORY_BRANCH], check=True)
        subprocess.run(["git", "-C", PROJECT_ROOT, "pull", "--ff-only", "origin", REPOSITORY_BRANCH], check=True)
    else:
        print("Using existing checkout (UPDATE_CODE=False).")
    subprocess.run(["git", "-C", PROJECT_ROOT, "log", "-1", "--oneline"], check=True)

## Restore state once per runtime

In [ ]:
import time as _time
_STATE_STAGE_STARTED = _time.time()

import os, shutil, sqlite3, sys, threading
from datetime import datetime, timezone
from pathlib import Path

sys.path.insert(0, f"{PROJECT_ROOT}/deployment")
import kaggle_persist


def _load_kaggle_credentials():
    if os.getenv("KAGGLE_USERNAME") and os.getenv("KAGGLE_KEY"):
        return True

    try:
        from kaggle_secrets import UserSecretsClient

        secrets = UserSecretsClient()
        os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")
        return True

    except Exception as error:
        print("Kaggle persistence credentials unavailable:", error)
        return False


def _configured_slug(slug):
    return bool(slug) and "YOUR_KAGGLE_USERNAME" not in slug


if (
    STATE_DATASET_SLUG == CACHE_DATASET_SLUG
    and _configured_slug(STATE_DATASET_SLUG)
):
    raise ValueError(
        "STATE_DATASET_SLUG and CACHE_DATASET_SLUG must be different private Kaggle Datasets."
    )


STAGING_DIR = Path("/kaggle/working/.persist")
CACHE_DIR = Path("/kaggle/working/.cache")
RUNTIME_MARKERS = Path("/kaggle/working/.ai-tutor-runtime-markers")

for directory in (STAGING_DIR, CACHE_DIR, RUNTIME_MARKERS):
    directory.mkdir(parents=True, exist_ok=True)


PERSISTENCE_ENABLED = (
    _configured_slug(STATE_DATASET_SLUG)
    and _load_kaggle_credentials()
)

CACHE_ENABLED = (
    _configured_slug(CACHE_DATASET_SLUG)
    and _load_kaggle_credentials()
)


def _restore_once(kind, slug, enabled, names, destination):
    marker = RUNTIME_MARKERS / f"{kind}-restore-attempted"

    if marker.exists():
        print(f"Skipping {kind} restore: already attempted in this runtime.")
        return

    # Only one restore attempt per runtime.
    marker.touch()

    if not enabled:
        print(
            f"{kind.title()} persistence disabled or its Dataset slug is still a placeholder."
        )
        return

    restore_dir = STAGING_DIR / f"{kind}_restore"

    if restore_dir.exists():
        shutil.rmtree(restore_dir)

    restore_dir.mkdir(parents=True)

    if not kaggle_persist.restore_dataset(slug, restore_dir):
        print(f"No saved {kind} snapshot found.")
        return

    for name in names:
        source = restore_dir / name

        if not source.exists():
            continue

        target = destination / name

        if target.exists():
            if target.is_dir():
                shutil.rmtree(target)
            else:
                target.unlink()

        shutil.move(str(source), str(target))

    print(f"Restored {kind} snapshot once for this runtime.")


_restore_once(
    "state",
    STATE_DATASET_SLUG,
    PERSISTENCE_ENABLED,
    ("data", "vectorstore"),
    Path(PROJECT_ROOT),
)

_restore_once(
    "cache",
    CACHE_DATASET_SLUG,
    CACHE_ENABLED,
    ("ollama-models", "pip", "npm"),
    CACHE_DIR,
)


for name in ("ollama-models", "pip", "npm"):
    (CACHE_DIR / name).mkdir(parents=True, exist_ok=True)


os.environ["OLLAMA_MODELS"] = str(CACHE_DIR / "ollama-models")
os.environ["PIP_CACHE_DIR"] = str(CACHE_DIR / "pip")
os.environ["NPM_CONFIG_CACHE"] = str(CACHE_DIR / "npm")


if "_STATE_SNAPSHOT_LOCK" not in globals():
    _STATE_SNAPSHOT_LOCK = threading.Lock()


def _is_sqlite_file(path):
    try:
        with path.open("rb") as handle:
            return handle.read(16) == b"SQLite format 3\x00"
    except OSError:
        return False


def _copy_live_tree(source, target):
    shutil.copytree(source, target)

    for source_file in source.rglob("*"):
        if not source_file.is_file():
            continue

        if not _is_sqlite_file(source_file):
            continue

        target_file = target / source_file.relative_to(source)
        temporary_db = target_file.with_name(target_file.name + ".backup")

        if temporary_db.exists():
            temporary_db.unlink()

        with sqlite3.connect(
            f"file:{source_file}?mode=ro",
            uri=True,
            timeout=30,
        ) as source_db:
            with sqlite3.connect(temporary_db) as backup_db:
                source_db.backup(backup_db)

        temporary_db.replace(target_file)


def save_state_snapshot(message="Active-session snapshot"):
    if not PERSISTENCE_ENABLED:
        print(
            "State persistence is disabled; configure STATE_DATASET_SLUG and Kaggle secrets."
        )
        return False

    if not _STATE_SNAPSHOT_LOCK.acquire(blocking=False):
        print("State snapshot already in progress; skipping duplicate request.")
        return False

    try:
        staging = STAGING_DIR / "state_save"
        temporary = STAGING_DIR / "state_save_building"

        for path in (temporary, staging):
            if path.exists():
                shutil.rmtree(path)

        temporary.mkdir(parents=True)

        for name in ("data", "vectorstore"):
            source = Path(PROJECT_ROOT) / name

            if source.exists():
                _copy_live_tree(source, temporary / name)

        temporary.replace(staging)

        kaggle_persist.save_dataset(
            STATE_DATASET_SLUG,
            staging,
            title="AI Tutor state",
            message=message,
        )

        print(
            "State snapshot completed at",
            datetime.now(timezone.utc).isoformat(),
        )

        return True

    except Exception as error:
        print(
            "State snapshot failed; the active runtime was left untouched:",
            error,
        )
        return False

    finally:
        _STATE_SNAPSHOT_LOCK.release()


STARTUP_TIMINGS["state restore"] = _time.time() - _STATE_STAGE_STARTED
print(f"[timing] state restore: {STARTUP_TIMINGS['state restore']:.1f}s")

In [ ]:
import hashlib, os, shutil, subprocess, sys, urllib.request
from pathlib import Path

CLOUDFLARED_BIN = Path("/usr/local/bin/cloudflared")
CLOUDFLARED_DOWNLOAD_URL = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"

with timed("dependencies"):
    # System packages: installed only when a binary is missing.
    required = [("nginx", "nginx"), ("curl", "curl"), ("zstd", "zstd")]
    missing = [package for package, binary in required if not shutil.which(binary)]
    if missing:
        # Kaggle may include a CRAN apt source that occasionally fails while its
        # mirror is synchronizing. This project does not require R/CRAN packages.
        source_dir = Path("/etc/apt/sources.list.d")
        if source_dir.exists():
            for source_file in source_dir.iterdir():
                if not source_file.is_file():
                    continue
                try:
                    text = source_file.read_text(errors="ignore")
                except Exception:
                    continue
                if "cloud.r-project.org" in text:
                    print(f"Temporarily disabling CRAN apt source: {source_file}")
                    source_file.rename(source_file.with_suffix(source_file.suffix + ".disabled"))
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", *missing], check=True)
    else:
        print("apt: nginx, curl, zstd already installed.")

    # Python / Node dependencies: skipped while requirements.txt / package-lock.json are unchanged.
    setup_markers = RUNTIME_MARKERS / "setup"
    setup_markers.mkdir(exist_ok=True)
    requirements = Path(PROJECT_ROOT) / "requirements.txt"
    pip_marker = setup_markers / f"pip-{hashlib.sha256(requirements.read_bytes()).hexdigest()}"
    if QUICK_START and pip_marker.exists():
        print("pip: requirements unchanged; skipping.")
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)
        pip_marker.touch()
    lockfile = Path(PROJECT_ROOT) / "frontend/package-lock.json"
    npm_marker = setup_markers / f"npm-{hashlib.sha256(lockfile.read_bytes()).hexdigest()}"
    if QUICK_START and npm_marker.exists():
        print("npm: lockfile unchanged; skipping.")
    else:
        subprocess.run(["npm", "ci", "--prefer-offline"], cwd=lockfile.parent, check=True)
        npm_marker.touch()

    # Ollama: installer runs only when the binary is missing.
    if shutil.which("ollama"):
        print("Ollama already installed:", shutil.which("ollama"))
    else:
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

    # cloudflared: downloaded only when missing.
    if shutil.which("cloudflared") or CLOUDFLARED_BIN.exists():
        print("cloudflared already installed:", shutil.which("cloudflared") or CLOUDFLARED_BIN)
    else:
        temporary = CLOUDFLARED_BIN.with_name("cloudflared.download")
        urllib.request.urlretrieve(CLOUDFLARED_DOWNLOAD_URL, temporary)
        temporary.chmod(0o755)
        os.replace(temporary, CLOUDFLARED_BIN)
        subprocess.run([str(CLOUDFLARED_BIN), "--version"], check=True)

In [ ]:
import hashlib, json, os, shutil, socket, subprocess, time, urllib.request
from pathlib import Path

# start_kaggle.sh owns the Ollama start, Qwen/bge-m3 pulls (skipped when present), the once-per-runtime
# warmup, the Chroma embedding-model check, incremental indexing, and FastAPI/frontend/nginx startup.
RUNTIME_DIR = Path("/kaggle/working/ai-tutor-runtime")
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
APP_FINGERPRINT_FILE = RUNTIME_DIR / "launcher-app-fingerprint"
# First log line of start_kaggle.sh after the Ollama/model phase (Chroma check, then indexing, then FastAPI).
INDEXING_PHASE_MARKERS = ("Chroma embedding marker", "Initialized Chroma", "WARNING: Embedding model", "Incrementally indexing", "Starting FastAPI")

data_dir = Path(PROJECT_ROOT) / "data"
data_dir.mkdir(parents=True, exist_ok=True)
(Path(PROJECT_ROOT) / "vectorstore").mkdir(parents=True, exist_ok=True)
copied_lectures = 0
if LECTURE_INPUT_DIR:
    source_dir = Path(LECTURE_INPUT_DIR)
    if not source_dir.is_dir(): raise FileNotFoundError(f"Lecture input not found: {source_dir}")
    for source in source_dir.rglob("*"):
        if source.is_file() and source.suffix.lower() in {".pdf", ".txt"}:
            target = data_dir / source.name
            if not target.exists() or target.stat().st_size != source.stat().st_size:
                shutil.copy2(source, target)
                copied_lectures += 1
print("Runtime data directory:", data_dir, f"(new lecture files: {copied_lectures})")


def _app_health_ok():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/api/health", timeout=5) as response:
            health = json.load(response)
    except Exception:
        return False
    return (
        health.get("status") == "ok"
        and health.get("chat_model") == OLLAMA_CHAT_MODEL
        and health.get("embedding_model") == OLLAMA_EMBEDDING_MODEL
    )


def _port_in_use(port):
    with socket.socket() as probe:
        probe.settimeout(1)
        return probe.connect_ex(("127.0.0.1", port)) == 0


head = subprocess.run(["git", "-C", PROJECT_ROOT, "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
app_fingerprint = hashlib.sha256(
    json.dumps({"head": head, "env": {key: os.environ.get(key, "") for key in APP_ENV}}, sort_keys=True).encode()
).hexdigest()

reuse_services = (
    not FORCE_RESTART
    and copied_lectures == 0
    and APP_FINGERPRINT_FILE.exists()
    and APP_FINGERPRINT_FILE.read_text().strip() == app_fingerprint
    and _app_health_ok()
)

if reuse_services:
    print("Services are healthy and code/config are unchanged; skipping restart (set FORCE_RESTART=True to restart).")
    STARTUP_TIMINGS["ollama/models"] = 0.0
    STARTUP_TIMINGS["indexing/app startup"] = 0.0
else:
    APP_FINGERPRINT_FILE.unlink(missing_ok=True)
    started = time.time()
    indexing_started = None
    process = subprocess.Popen(
        ["bash", f"{PROJECT_ROOT}/deployment/start_kaggle.sh"],
        cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in process.stdout:
        if indexing_started is None and any(marker in line for marker in INDEXING_PHASE_MARKERS):
            indexing_started = time.time()
        print(line, end="", flush=True)
    return_code = process.wait()
    finished = time.time()
    indexing_started = indexing_started or finished
    STARTUP_TIMINGS["ollama/models"] = indexing_started - started
    STARTUP_TIMINGS["indexing/app startup"] = finished - indexing_started
    if return_code:
        raise subprocess.CalledProcessError(return_code, "deployment/start_kaggle.sh")
    APP_FINGERPRINT_FILE.write_text(app_fingerprint)
    print("Services restarted and validated by start_kaggle.sh.")

for stage in ("ollama/models", "indexing/app startup"):
    print(f"[timing] {stage}: {STARTUP_TIMINGS[stage]:.1f}s")
if _port_in_use(8001):
    print("WARNING: something is listening on 127.0.0.1:8001 (probably a vLLM server left by an earlier vLLM run in this runtime). This launcher does not use it.")

In [ ]:
# Start one periodic state saver per Kaggle runtime after the app is running.
def _periodic_state_saver():
    while not _STATE_SNAPSHOT_STOP.wait(STATE_SNAPSHOT_INTERVAL_SECONDS):
        save_state_snapshot(message="Automatic 10-minute snapshot")

if not PERSISTENCE_ENABLED:
    print("Periodic state snapshots are disabled until STATE_DATASET_SLUG and Kaggle secrets are configured.")
elif globals().get("_STATE_SNAPSHOT_THREAD") and _STATE_SNAPSHOT_THREAD.is_alive():
    print("Periodic state snapshot worker is already running in this runtime.")
else:
    _STATE_SNAPSHOT_STOP = threading.Event()
    _STATE_SNAPSHOT_THREAD = threading.Thread(
        target=_periodic_state_saver,
        name="ai-tutor-state-snapshot",
        daemon=True,
    )
    _STATE_SNAPSHOT_THREAD.start()
    print(
        f"Periodic state snapshots enabled every "
        f"{STATE_SNAPSHOT_INTERVAL_SECONDS // 60} minutes."
    )

In [ ]:
import json, os, re, shutil, signal, subprocess, time, urllib.error, urllib.request
from pathlib import Path
from urllib.parse import urlparse

TUNNEL_TARGET = f"http://127.0.0.1:{PUBLIC_PORT}"  # nginx, which serves both the frontend and /api/
CLOUDFLARED = shutil.which("cloudflared") or "/usr/local/bin/cloudflared"
CLOUDFLARED_PID_FILE = RUNTIME_DIR / "cloudflared.pid"
CLOUDFLARED_URL_FILE = RUNTIME_DIR / "cloudflared.url"
CLOUDFLARED_LOG = Path("/kaggle/working/ai-tutor-logs/cloudflared.log")
QUICK_TUNNEL_URL = re.compile(r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com")

# Permanent Worker URL; the Worker proxies to the origin stored in Workers KV key ORIGIN_URL.
GATEWAY_URL = "https://ai-tutor-gateway.quoctrung121212.workers.dev"
GATEWAY_KV_KEY = "ORIGIN_URL"


def _cloudflared_alive(pid):
    try:
        cmdline = Path(f"/proc/{pid}/cmdline").read_bytes()
        state = Path(f"/proc/{pid}/stat").read_text().rsplit(")", 1)[1].split()[0]
    except (OSError, IndexError):
        return False
    return b"cloudflared" in cmdline and state != "Z"


def _terminate(pid):
    os.kill(pid, signal.SIGTERM)
    for _ in range(20):
        if not _cloudflared_alive(pid):
            return
        time.sleep(0.25)
    os.kill(pid, signal.SIGKILL)


def _update_gateway_origin(origin_url):
    """PUT origin_url into Workers KV. Credentials come only from Kaggle Secrets and are never printed."""
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        token = (secrets.get_secret("CLOUDFLARE_API_TOKEN") or "").strip()
        account_id = (secrets.get_secret("CLOUDFLARE_ACCOUNT_ID") or "").strip()
        namespace_id = (secrets.get_secret("CLOUDFLARE_KV_NAMESPACE_ID") or "").strip()
    except Exception as error:
        print(f"[cloudflare-gateway] Could not read Cloudflare Kaggle Secrets ({type(error).__name__}).")
        return False
    if not (token and account_id and namespace_id):
        print("[cloudflare-gateway] Missing Kaggle Secret CLOUDFLARE_API_TOKEN, CLOUDFLARE_ACCOUNT_ID or CLOUDFLARE_KV_NAMESPACE_ID.")
        return False

    def redact(text):
        for secret in (token, account_id, namespace_id):
            text = text.replace(secret, "***")
        return text

    try:
        request = urllib.request.Request(
            f"https://api.cloudflare.com/client/v4/accounts/{account_id}/storage/kv/namespaces/{namespace_id}/values/{GATEWAY_KV_KEY}",
            data=origin_url.encode(),
            method="PUT",
            headers={"Authorization": f"Bearer {token}", "Content-Type": "text/plain", "User-Agent": "ai-tutor-kaggle-launcher"},
        )
        with urllib.request.urlopen(request, timeout=20) as response:
            status, body = response.status, response.read().decode(errors="ignore")
    except urllib.error.HTTPError as error:
        status, body = error.code, error.read().decode(errors="ignore")
    except Exception as error:
        print(f"[cloudflare-gateway] KV update request failed: {redact(f'{type(error).__name__}: {error}')}")
        return False

    try:
        payload = json.loads(body)
    except ValueError:
        payload = {}
    if not isinstance(payload, dict):
        payload = {}
    if status == 200 and payload.get("success", True):
        return True
    messages = "; ".join(
        f"{item.get('code')}: {item.get('message')}" for item in payload.get("errors") or [] if isinstance(item, dict)
    )
    print(f"[cloudflare-gateway] KV update failed: HTTP {status} {redact(messages or body[:300])}")
    return False


with timed("cloudflare"):
    if not Path(CLOUDFLARED).exists():
        raise RuntimeError("cloudflared is missing; run the dependencies cell first.")
    old_pid_text = CLOUDFLARED_PID_FILE.read_text().strip() if CLOUDFLARED_PID_FILE.exists() else ""
    old_pid = int(old_pid_text) if old_pid_text.isdigit() else None
    old_url = CLOUDFLARED_URL_FILE.read_text().strip() if CLOUDFLARED_URL_FILE.exists() else ""

    if old_pid and _cloudflared_alive(old_pid) and old_url:
        PUBLIC_URL = old_url
        print(f"Reusing running Cloudflare tunnel (PID {old_pid}).")
    else:
        if old_pid and _cloudflared_alive(old_pid):
            print(f"Terminating old cloudflared process {old_pid} (its URL is unknown).")
            _terminate(old_pid)
        CLOUDFLARED_PID_FILE.unlink(missing_ok=True)
        CLOUDFLARED_URL_FILE.unlink(missing_ok=True)
        CLOUDFLARED_LOG.parent.mkdir(parents=True, exist_ok=True)
        with CLOUDFLARED_LOG.open("w") as log_handle:
            tunnel_process = subprocess.Popen(
                [CLOUDFLARED, "tunnel", "--url", TUNNEL_TARGET, "--no-autoupdate"],
                stdout=log_handle, stderr=subprocess.STDOUT, start_new_session=True,
            )
        CLOUDFLARED_PID_FILE.write_text(str(tunnel_process.pid))

        PUBLIC_URL = ""
        deadline = time.time() + 90
        while time.time() < deadline:
            log_text = CLOUDFLARED_LOG.read_text(errors="ignore")
            match = QUICK_TUNNEL_URL.search(log_text)
            if match and not PUBLIC_URL:
                PUBLIC_URL = match.group(0)
            if PUBLIC_URL and "Registered tunnel connection" in log_text:
                break
            if tunnel_process.poll() is not None:
                raise RuntimeError(f"cloudflared exited with code {tunnel_process.returncode}:\n{log_text[-3000:]}")
            time.sleep(1)
        if not PUBLIC_URL:
            raise RuntimeError(f"No trycloudflare.com URL within 90s:\n{CLOUDFLARED_LOG.read_text(errors='ignore')[-3000:]}")
        CLOUDFLARED_URL_FILE.write_text(PUBLIC_URL)

    # Best effort: Kaggle DNS can lag behind a brand-new quick-tunnel hostname.
    PUBLIC_HEALTH_OK = False
    last_error = None
    for _ in range(10):
        try:
            with urllib.request.urlopen(f"{PUBLIC_URL}/api/health", timeout=5) as response:
                print("Public health check:", response.status)
                PUBLIC_HEALTH_OK = response.status == 200
                break
        except Exception as error:
            last_error = error
            time.sleep(2)
    else:
        print("Public health check not confirmed from inside Kaggle yet:", last_error)

# Point the permanent Worker at this tunnel. Failures here never touch the tunnel or app services.
with timed("cloudflare gateway"):
    GATEWAY_UPDATED = False
    if not PUBLIC_HEALTH_OK:
        print(f"[cloudflare-gateway] Skipping {GATEWAY_KV_KEY} update: {PUBLIC_URL}/api/health did not return 200.")
    else:
        GATEWAY_UPDATED = _update_gateway_origin(PUBLIC_URL)

    if GATEWAY_UPDATED:
        print(f"[cloudflare-gateway] {GATEWAY_KV_KEY} updated")
        # Workers KV is eventually consistent, so the Worker may serve the old origin for up to ~60s.
        gateway_deadline = time.time() + 60
        gateway_error = None
        while True:
            try:
                with urllib.request.urlopen(f"{GATEWAY_URL}/api/health", timeout=10) as response:
                    print("[cloudflare-gateway] Gateway health check:", response.status)
                    break
            except Exception as error:
                gateway_error = error
            if time.time() >= gateway_deadline:
                print("[cloudflare-gateway] Gateway health not confirmed within 60s (KV may still be propagating):", gateway_error)
                break
            time.sleep(5)

print("\n" + "=" * 72)
if GATEWAY_UPDATED:
    print("  Permanent AI Tutor URL:")
    print(f"  {GATEWAY_URL}")
    print()
else:
    print(f"  Permanent gateway NOT updated; {GATEWAY_URL} may still point at an old tunnel.")
print(f"  Temporary tunnel URL:  {PUBLIC_URL}")
print(f"  (Cloudflare Quick Tunnel -> nginx {TUNNEL_TARGET})")
print("=" * 72 + "\n")

frontend_url = os.environ.get("AI_TUTOR_FRONTEND_URL", "")
if frontend_url and urlparse(frontend_url).hostname != urlparse(PUBLIC_URL).hostname:
    print(
        "Note: Google Calendar OAuth redirects use the Kaggle Secrets GOOGLE_CALENDAR_REDIRECT_URI / "
        f"AI_TUTOR_FRONTEND_URL ({frontend_url}), not this tunnel URL."
    )

In [ ]:
STAGES = ("git", "state restore", "dependencies", "ollama/models", "indexing/app startup", "cloudflare")
print("Startup timing")
for stage in STAGES:
    value = STARTUP_TIMINGS.get(stage)
    print(f"  {stage:<22}{'not run' if value is None else f'{value:7.1f}s'}")
print(f"  {'total (wall clock)':<22}{time.time() - RUN_STARTED:7.1f}s")

## Manual actions

These cells do nothing during a normal **Run All**. Set the flag in a cell and run that cell on its own.

Automatic snapshots run about every 10 minutes after services start. As a backup, set `SAVE_STATE_NOW=True` and run the cell before ending a session. Both `data/` and `vectorstore/` are persisted.

In [ ]:
SAVE_STATE_NOW = False

if not SAVE_STATE_NOW:
    print(
        "Automatic snapshots are active when persistence is configured. "
        "Set SAVE_STATE_NOW=True and rerun for a manual backup."
    )
else:
    save_state_snapshot(message="Manual active-session snapshot")

In [ ]:
SAVE_CACHE_NOW = False
if not SAVE_CACHE_NOW: print("Nothing saved. Set SAVE_CACHE_NOW=True and rerun this cell.")
elif not CACHE_ENABLED: print("Cache persistence is disabled.")
else:
    staging = STAGING_DIR / "cache_save"
    if staging.exists(): shutil.rmtree(staging)
    staging.mkdir(parents=True)
    for name in ("ollama-models", "pip", "npm"):
        source = CACHE_DIR / name
        if source.exists() and any(source.iterdir()): shutil.copytree(source, staging / name)
    kaggle_persist.save_dataset(CACHE_DATASET_SLUG, staging, title="AI Tutor build cache", message="Updated runtime cache")
    print("Cache saved.")

In [ ]:
import subprocess, sys
# Tests are not part of a normal Run All; set QUICK_START=False in the config cell to run them.
if QUICK_START:
    print("QUICK_START=True: skipping tests.")
else:
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=PROJECT_ROOT, check=True)

In [ ]:
RUN_DIAGNOSTICS = False

if not RUN_DIAGNOSTICS:
    print("Diagnostics skipped. Set RUN_DIAGNOSTICS=True and rerun this cell.")
else:
    import subprocess
    # Expected listeners: 11434 ollama, 8000 backend, 3000 frontend, 7860 nginx; 8001 (vLLM) should be absent.
    subprocess.run("ss -ltnp | grep -E ':11434|:8001|:8000|:3000|:7860' || true", shell=True)
    subprocess.run(["ollama", "ps"])
    subprocess.run("tail -n 40 /kaggle/working/ai-tutor-logs/backend.log /kaggle/working/ai-tutor-logs/cloudflared.log", shell=True)